**atividade pratica:**

Modifique o bloco acima para contabilizar o total de passos (time steps), e use isso como métrica pra avaliar o quão bom está o nosso sistema de Q-learning. Talvez seja interessante aumentar o número de viagens simuladas, e remover as chamadas de sleep() pra conseguir rodar em mais amostras.

Agora, experimente com os hiperparâmetros. O quão baixo o número de epochs pode ir antes do nosso modelo começar a sofrer? Você consegue chegar em learning rates, discount factors ou exploration factors melhores pra deixar o treinamento mais eficiente? A taxa de exploração vs. explotação em particular é interessante de se experimentar.

In [1]:
# Mesma preparação da aula
# OBS: removi o ".env" do gym.make. Com ".env" o episódio perde o limite de 200 passos,
# e com poucos epochs (política ainda ruim) o episódio pode nunca terminar e travar o notebook.
import warnings
warnings.filterwarnings('ignore')  # só pra não poluir a saída com avisos de versão do gym

import gym
import random
import numpy as np

random.seed(1234)
streets = gym.make("Taxi-v3")

In [2]:
# Treino original empacotado em função, recebendo os hiperparâmetros como argumento
def train_q_table(env, learning_rate, discount_factor, exploration, epochs):
    q_table = np.zeros([env.observation_space.n, env.action_space.n])

    for taxi_run in range(epochs):
        state = env.reset()
        done = False

        while not done:
            random_value = random.uniform(0, 1)
            if random_value < exploration:
                action = env.action_space.sample()  # explora uma ação aleatória
            else:
                action = np.argmax(q_table[state])  # usa a ação com maior q-value

            next_state, reward, done, info = env.step(action)

            prev_q = q_table[state, action]
            next_max_q = np.max(q_table[next_state])
            new_q = (1 - learning_rate) * prev_q + learning_rate * (reward + discount_factor * next_max_q)
            q_table[state, action] = new_q

            state = next_state

    return q_table

**1. Contando os passos, sem sleep(), com mais viagens**

In [3]:
# Roda N viagens usando a q_table já treinada e conta quantos passos cada uma levou
# sem sleep() e sem render, só pra medir desempenho
def evaluate_q_table(env, q_table, num_trips):
    total_steps = 0

    for tripnum in range(num_trips):
        state = env.reset()
        done = False
        steps = 0

        while not done:
            action = np.argmax(q_table[state])  # sempre explota, sem exploração aqui
            next_state, reward, done, info = env.step(action)
            steps += 1
            state = next_state

        total_steps += steps

    return total_steps, total_steps / num_trips

In [4]:
# Treino padrão (mesmos hiperparâmetros da aula) e avaliação com 1000 viagens
learning_rate = 0.1
discount_factor = 0.6
exploration = 0.1
epochs = 10000

q_table = train_q_table(streets, learning_rate, discount_factor, exploration, epochs)

num_trips = 1000
total_steps, avg_steps = evaluate_q_table(streets, q_table, num_trips)
print(f'Total de passos em {num_trips} viagens: {total_steps}')
print(f'Média de passos por viagem: {avg_steps:.2f}')

Total de passos em 1000 viagens: 31898
Média de passos por viagem: 31.90


**2. Experimentando com epochs**

In [5]:
# Testa vários valores de epochs, treinando do zero pra cada um,
# e avalia a média de passos pra ver a partir de que ponto o modelo piora
epoch_values = [50, 100, 250, 500, 1000, 2500, 5000, 10000]

for e in epoch_values:
    q_table_test = train_q_table(streets, learning_rate=0.1, discount_factor=0.6, exploration=0.1, epochs=e)
    _, avg = evaluate_q_table(streets, q_table_test, num_trips=200)
    print(f'epochs = {e:6d} -> média de passos: {avg:.2f}')

epochs =     50 -> média de passos: 200.00
epochs =    100 -> média de passos: 200.00
epochs =    250 -> média de passos: 200.00
epochs =    500 -> média de passos: 198.06
epochs =   1000 -> média de passos: 158.11
epochs =   2500 -> média de passos: 93.59
epochs =   5000 -> média de passos: 70.38
epochs =  10000 -> média de passos: 27.90


**3. Experimentando learning rate, discount factor e exploration**

In [6]:
# Varia o learning rate mantendo o resto fixo
for lr in [0.05, 0.1, 0.3, 0.5, 0.9]:
    q_table_test = train_q_table(streets, learning_rate=lr, discount_factor=0.6, exploration=0.1, epochs=5000)
    _, avg = evaluate_q_table(streets, q_table_test, num_trips=200)
    print(f'learning_rate = {lr:.2f} -> média de passos: {avg:.2f}')

learning_rate = 0.05 -> média de passos: 107.64
learning_rate = 0.10 -> média de passos: 66.47
learning_rate = 0.30 -> média de passos: 19.28
learning_rate = 0.50 -> média de passos: 13.02
learning_rate = 0.90 -> média de passos: 13.30


In [7]:
# Varia o discount factor mantendo o resto fixo
for df in [0.1, 0.3, 0.6, 0.9, 0.99]:
    q_table_test = train_q_table(streets, learning_rate=0.1, discount_factor=df, exploration=0.1, epochs=5000)
    _, avg = evaluate_q_table(streets, q_table_test, num_trips=200)
    print(f'discount_factor = {df:.2f} -> média de passos: {avg:.2f}')

discount_factor = 0.10 -> média de passos: 113.19
discount_factor = 0.30 -> média de passos: 104.94
discount_factor = 0.60 -> média de passos: 50.83
discount_factor = 0.90 -> média de passos: 12.96
discount_factor = 0.99 -> média de passos: 12.98


In [8]:
# Varia a taxa de exploração mantendo o resto fixo (aqui é onde exploração vs explotação pesa mais)
for exp in [0.01, 0.05, 0.1, 0.3, 0.5, 0.8]:
    q_table_test = train_q_table(streets, learning_rate=0.1, discount_factor=0.6, exploration=exp, epochs=5000)
    _, avg = evaluate_q_table(streets, q_table_test, num_trips=200)
    print(f'exploration = {exp:.2f} -> média de passos: {avg:.2f}')

exploration = 0.01 -> média de passos: 58.09
exploration = 0.05 -> média de passos: 56.41
exploration = 0.10 -> média de passos: 63.76
exploration = 0.30 -> média de passos: 55.38
exploration = 0.50 -> média de passos: 32.25
exploration = 0.80 -> média de passos: 15.86


 **4. Melhor combinação encontrada**

In [9]:
# Junta os melhores valores observados nos testes acima e roda a avaliação final
best_learning_rate = 0.3
best_discount_factor = 0.9
best_exploration = 0.1
best_epochs = 5000

q_table_best = train_q_table(streets, best_learning_rate, best_discount_factor, best_exploration, best_epochs)
total_steps, avg_steps = evaluate_q_table(streets, q_table_best, num_trips=1000)
print(f'Total de passos em 1000 viagens: {total_steps}')
print(f'Média de passos por viagem: {avg_steps:.2f}')

Total de passos em 1000 viagens: 12975
Média de passos por viagem: 12.97
